# JSON SFT Warm-Up + GRPO for Flatmate RL

This notebook combines two stages:

1. **JSON SFT warm-up**: teach the model to emit valid Flatmate RL JSON actions from endpoint observations.
2. **GRPO**: continue from the saved SFT adapter using strict rewards. Invalid JSON is penalized; it is never replaced by the heuristic during training.

The GRPO rewards include format/schema rewards, reference-action rewards, and live endpoint transition rewards. This avoids the common failure mode where all completions receive the same reward and GRPO advantages become zero.

In [ ]:
# Restart the kernel after this cell if your runtime asks you to.
%pip install -q "trl>=0.23.0" "transformers>=4.46.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas

In [ ]:
from __future__ import annotations

import asyncio
import inspect
import json
import random
import re
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
import websockets
from datasets import Dataset

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_OUTPUT_DIR = "flatmate-rl-json-sft-policy"
SFT_MERGED_DIR = "flatmate-rl-json-sft-merged"
GRPO_OUTPUT_DIR = "flatmate-rl-json-grpo-policy"


def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"


SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL

## Endpoint Client

Use `/ws` because OpenEnv keeps multi-step episode state inside a websocket session.

In [ ]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(
            self.ws_url,
            open_timeout=self.timeout_s,
            ping_timeout=self.timeout_s,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})


async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=901)
        print(obs["scenario_id"], obs["status"])
        print(obs.get("last_user_message") or obs.get("current_user_request"))


await smoke_test_endpoint()

## Prompts, Parsing, and Heuristic Demonstrations

The heuristic is used only to collect demonstrations and reference actions. It is not used as a fallback during GRPO reward scoring.

In [ ]:
def trace_tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]


def heuristic_action(obs: dict[str, Any]) -> dict[str, Any] | None:
    tools = trace_tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")

    if phase == "seller" and not obs.get("seller_profile_stored"):
        if remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    post_ids = ["post_031", "post_052"] if scenario_id == "task_visit_multi" else ["post_031"]
    if "search_posts" not in tools:
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}
    if "match_location_preference" not in tools:
        return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": post_ids}}
    if "get_commute_time" not in tools:
        return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": post_ids}}
    if "check_calendar_slots" not in tools:
        return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": post_ids}}
    if "shortlist" not in tools:
        return {"action_type": "tool_call", "tool_name": "shortlist", "tool_arguments": {"post_ids": post_ids}}
    if "contact_poster" not in tools:
        return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}
    if "book_viewing" not in tools:
        return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}
    return None


def compact_observation(obs: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }


def prompt_from_observation(obs: dict[str, Any]) -> str:
    action_shape_message = json.dumps({
        "action_type": "assistant_message",
        "assistant_message": "...",
    }, separators=(",", ":"))
    action_shape_tool = '{"action_type":"tool_call","tool_name":"...","tool_arguments":{}}'
    observation_json = json.dumps(compact_observation(obs), ensure_ascii=False, sort_keys=True)
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Return exactly one minified JSON action and no markdown.\n"
        "Valid shapes:\n"
        f"{action_shape_message}\n"
        f"{action_shape_tool}\n\n"
        f"Observation:\n{observation_json}\n\n"
        "Action:\n"
    )

def make_sft_text(obs: dict[str, Any], action: dict[str, Any]) -> str:
    return prompt_from_observation(obs) + json.dumps(action, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def first_balanced_json(text: str) -> str:
    start = text.find("{")
    if start == -1:
        raise ValueError("no JSON object found")
    depth = 0
    in_string = False
    escape = False
    for index, char in enumerate(text[start:], start=start):
        if escape:
            escape = False
            continue
        if char == "\\" and in_string:
            escape = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start : index + 1]
    raise ValueError("unterminated JSON object")


def normalize_action(action: dict[str, Any]) -> dict[str, Any]:
    if action.get("action_type") == "assistant_message":
        message = str(action.get("assistant_message", "")).strip()
        if not message:
            raise ValueError("assistant_message action needs assistant_message")
        return {"action_type": "assistant_message", "assistant_message": message}
    if action.get("action_type") == "tool_call":
        tool_name = str(action.get("tool_name", "")).strip()
        if not tool_name:
            raise ValueError("tool_call action needs tool_name")
        tool_arguments = action.get("tool_arguments", {})
        if not isinstance(tool_arguments, dict):
            raise ValueError("tool_arguments must be an object")
        return {"action_type": "tool_call", "tool_name": tool_name, "tool_arguments": tool_arguments}
    raise ValueError("action_type must be assistant_message or tool_call")


def parse_json_action(text: str) -> dict[str, Any]:
    return normalize_action(json.loads(first_balanced_json(text)))


def canonical_action(action: dict[str, Any]) -> str:
    return json.dumps(normalize_action(action), ensure_ascii=False, sort_keys=True, separators=(",", ":"))

## Balanced Episode-Level Data

Collect train and test episodes for every task ID. Seeds create value variants after the updated Space code is uploaded.

In [ ]:
@dataclass
class RolloutConfig:
    train_episodes_per_task: int = 4
    test_episodes_per_task: int = 2
    max_steps: int = 20
    seed: int = 7


async def collect_one_episode(scenario_id: str, episode_id: str, split: str, seed: int, max_steps: int) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    prefix_actions: list[dict[str, Any]] = []
    async with FlatmateEndpoint() as env:
        obs = await env.reset(scenario_id, seed=seed)
        total_reward = 0.0
        for step_idx in range(max_steps):
            action = heuristic_action(obs)
            if action is None or obs.get("done"):
                break
            completion = canonical_action(action)
            rows.append({
                "prompt": prompt_from_observation(obs),
                "completion": completion,
                "text": make_sft_text(obs, action),
                "episode_id": episode_id,
                "split": split,
                "scenario_id": scenario_id,
                "seed": seed,
                "step": step_idx,
                "prefix_actions_json": json.dumps(prefix_actions, ensure_ascii=False),
                "reference_action_json": completion,
            })
            obs = await env.step(action)
            prefix_actions.append(action)
            total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
            if obs.get("done"):
                break
    print(f"split={split:5s} episode={episode_id} scenario={scenario_id} rows={len(rows)} total_reward={total_reward:.2f}")
    return rows


async def collect_balanced_rollouts(config: RolloutConfig = RolloutConfig()) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    train_rows: list[dict[str, Any]] = []
    test_rows: list[dict[str, Any]] = []
    for scenario_idx, scenario_id in enumerate(SCENARIOS):
        for i in range(config.train_episodes_per_task):
            seed = config.seed + scenario_idx * 100 + i
            train_rows.extend(await collect_one_episode(scenario_id, f"train_{scenario_id}_{i:03d}", "train", seed, config.max_steps))
        for i in range(config.test_episodes_per_task):
            seed = 900 + config.seed + scenario_idx * 100 + i
            test_rows.extend(await collect_one_episode(scenario_id, f"test_{scenario_id}_{i:03d}", "test", seed, config.max_steps))
    return train_rows, test_rows


train_rows, test_rows = await collect_balanced_rollouts(RolloutConfig())
train_dataset = Dataset.from_list(train_rows)
test_dataset = Dataset.from_list(test_rows)
grpo_dataset = Dataset.from_list(test_rows + train_rows).shuffle(seed=42)

summary = {
    "train_rows": len(train_dataset),
    "test_rows": len(test_dataset),
    "train_episodes": len(set(train_dataset["episode_id"])),
    "test_episodes": len(set(test_dataset["episode_id"])),
}
print(summary)
display(pd.DataFrame(train_rows).groupby("scenario_id")["episode_id"].nunique().rename("train_episodes"))
display(pd.DataFrame(test_rows).groupby("scenario_id")["episode_id"].nunique().rename("test_episodes"))

## Stage 1: SFT Warm-Up for Valid JSON

This stage is not the final policy optimization. Its job is to make valid JSON actions common enough that GRPO has non-identical rewards to learn from.

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, device_map="auto")
base_model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_args = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1536,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    packing=False,
    report_to="none",
    remove_unused_columns=False,
)

sft_trainer = SFTTrainer(
    model=base_model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

sft_result = sft_trainer.train()
sft_metrics = sft_trainer.evaluate()
sft_log_history = sft_trainer.state.log_history
sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print("sft_metrics", sft_metrics)
sft_result

In [ ]:
def plot_sft_log(log_history):
    df = pd.DataFrame(log_history)
    cols = [c for c in ["loss", "eval_loss"] if c in df.columns]
    if not cols or "step" not in df.columns:
        print("No SFT loss rows to plot yet.")
        return df
    ax = df.dropna(subset=["step"]).plot(x="step", y=cols, marker="o", figsize=(8, 4), title="SFT warm-up log")
    ax.grid(True, alpha=0.3)
    plt.show()
    return df

sft_log_df = plot_sft_log(sft_log_history)
sft_log_df.tail()

## SFT JSON Sanity Check

GRPO should start only after the saved SFT adapter can emit valid JSON at least sometimes. This cell reports the valid JSON rate on held-out prompts.

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM


def load_sft_model():
    model = AutoPeftModelForCausalLM.from_pretrained(SFT_OUTPUT_DIR, device_map="auto")
    model.eval()
    model.config.use_cache = False
    return model


sft_model_for_check = load_sft_model()


def generate_completion(model, prompt: str, max_new_tokens: int = 96, do_sample: bool = False) -> str:
    inputs = tokenizer(prompt + "{", return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    model.generation_config.do_sample = do_sample
    model.generation_config.temperature = 0.7 if do_sample else None
    model.generation_config.top_p = 0.9 if do_sample else None
    model.generation_config.top_k = None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            repetition_penalty=1.12,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return "{" + tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def json_sanity_for_dataset(model, dataset, limit: int = 24):
    rows = []
    for row in dataset.select(range(min(limit, len(dataset)))):
        raw = generate_completion(model, row["prompt"], do_sample=False)
        try:
            action = parse_json_action(raw)
            ok = True
            error = ""
        except Exception as exc:
            action = None
            ok = False
            error = str(exc)
        rows.append({
            "scenario_id": row["scenario_id"],
            "json_ok": ok,
            "raw": raw[:240],
            "parsed_action": action,
            "reference_action": row["reference_action_json"],
            "error": error,
        })
    df = pd.DataFrame(rows)
    print("json_ok_rate", df["json_ok"].mean() if len(df) else 0.0)
    return df


sft_json_sanity_df = json_sanity_for_dataset(sft_model_for_check, test_dataset, limit=32)
display(sft_json_sanity_df)

## Merge SFT Adapter for GRPO

Some TRL versions expect `model=` to point to a full model directory, not a PEFT adapter directory. Merge the SFT adapter into the base model before GRPO.

In [ ]:
from peft import AutoPeftModelForCausalLM

sft_adapter_model = AutoPeftModelForCausalLM.from_pretrained(SFT_OUTPUT_DIR, device_map="auto")
sft_merged_model = sft_adapter_model.merge_and_unload()
sft_merged_model.save_pretrained(SFT_MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(SFT_MERGED_DIR)
print(f"Saved merged SFT model to {SFT_MERGED_DIR}")

## GRPO Rewards

The reward is deliberately shaped so it usually has variance within each generated group:

- invalid JSON: strong negative
- valid action schema: positive
- same action type/tool as reference: positive
- exact reference action: bigger positive
- live endpoint transition: environment reward plus booking/violation shaping

No heuristic fallback is used during reward scoring.

In [ ]:
def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get("content", ""))
    return str(completion)


def invalid_json_progress_reward(text: str) -> float:
    # Deterministic partial credit keeps malformed generations from all receiving
    # exactly the same reward, which helps avoid zero-advantage GRPO groups early on.
    reward = -2.0
    stripped = text.strip()
    if "{" in stripped:
        reward += 0.15
    if "}" in stripped:
        reward += 0.15
    if "action_type" in stripped:
        reward += 0.25
    if "tool_call" in stripped or "assistant_message" in stripped:
        reward += 0.2
    if "tool_name" in stripped or "tool_arguments" in stripped:
        reward += 0.15
    if len(stripped) > 20:
        reward += 0.05
    if re.search(r"(\b\w+\b)(?:\s+\1){4,}", stripped):
        reward -= 0.4
    return float(max(-2.5, min(-0.2, reward)))


def reward_format_schema(completions, reference_action_json=None, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        text = completion_to_text(completion)
        try:
            action = parse_json_action(text)
        except Exception:
            rewards.append(invalid_json_progress_reward(text))
            continue
        reward = 0.4
        if action["action_type"] == "tool_call":
            reward += 0.2
            if isinstance(action.get("tool_arguments"), dict):
                reward += 0.1
        else:
            reward += 0.1
        rewards.append(reward)
    return rewards


def reward_reference_match(completions, reference_action_json, **kwargs) -> list[float]:
    rewards = []
    for completion, ref_json in zip(completions, reference_action_json):
        try:
            action = parse_json_action(completion_to_text(completion))
            ref = normalize_action(json.loads(ref_json))
        except Exception:
            rewards.append(-1.0)
            continue
        reward = 0.0
        if action["action_type"] == ref["action_type"]:
            reward += 0.4
        if action["action_type"] == "tool_call" and ref["action_type"] == "tool_call":
            if action.get("tool_name") == ref.get("tool_name"):
                reward += 0.8
            if canonical_action(action) == canonical_action(ref):
                reward += 0.8
        elif canonical_action(action) == canonical_action(ref):
            reward += 1.2
        rewards.append(reward)
    return rewards


async def score_endpoint_one(completion: Any, scenario_id: str, seed: int, prefix_actions_json: str) -> float:
    try:
        action = parse_json_action(completion_to_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -2.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get("done"):
                    return -0.5

            before_violations = len(obs.get("violations", []))
            before_bookings = len(obs.get("booked_visits", []))
            obs = await env.step(action)

        reward = float(obs.get("reward") or obs.get("step_reward") or 0.0)
        reward += 0.2
        if len(obs.get("violations", [])) > before_violations:
            reward -= 0.8
        if len(obs.get("booked_visits", [])) > before_bookings:
            reward += 1.2
        if obs.get("done") and len(obs.get("booked_visits", [])) > 0:
            reward += 1.0
        elif obs.get("done"):
            reward -= 0.5
        return float(max(-3.0, min(3.0, reward)))
    except Exception as exc:
        print("endpoint reward error:", repr(exc))
        return -1.5


async def score_endpoint_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    tasks = [score_endpoint_one(c, s, int(sd), p) for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)]
    return list(await asyncio.gather(*tasks))


def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}
    def runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            result["error"] = exc
    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


def reward_endpoint_transition(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    return run_async_blocking(score_endpoint_batch(completions, scenario_id, seed, prefix_actions_json))


# Smoke test rewards on reference actions. These should not all be zero.
sample = grpo_dataset.select(range(min(4, len(grpo_dataset))))
print("format", reward_format_schema(sample["reference_action_json"], reference_action_json=sample["reference_action_json"]))
print("reference", reward_reference_match(sample["reference_action_json"], reference_action_json=sample["reference_action_json"]))
print("endpoint", reward_endpoint_transition(
    completions=sample["reference_action_json"],
    scenario_id=sample["scenario_id"],
    seed=sample["seed"],
    prefix_actions_json=sample["prefix_actions_json"],
))

## Stage 2: GRPO from SFT Adapter

This uses the saved SFT adapter as the initial policy. Config kwargs are filtered against your installed TRL version to avoid version-specific crashes.

In [ ]:
from trl import GRPOConfig, GRPOTrainer


def make_grpo_config(**kwargs):
    valid = set(inspect.signature(GRPOConfig.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported GRPOConfig fields for this TRL version:", skipped)
    return GRPOConfig(**filtered)


def make_grpo_trainer(**kwargs):
    valid = set(inspect.signature(GRPOTrainer.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported GRPOTrainer fields for this TRL version:", skipped)
    return GRPOTrainer(**filtered)


grpo_args = make_grpo_config(
    output_dir=GRPO_OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1536,
    max_completion_length=160,
    max_steps=40,
    temperature=0.8,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

grpo_trainer = make_grpo_trainer(
    model=SFT_MERGED_DIR,
    reward_funcs=[reward_format_schema, reward_reference_match, reward_endpoint_transition],
    args=grpo_args,
    train_dataset=grpo_dataset,
)

grpo_result = grpo_trainer.train()
grpo_log_history = grpo_trainer.state.log_history
grpo_trainer.save_model(GRPO_OUTPUT_DIR)
grpo_result

In [ ]:
def plot_grpo_log(log_history):
    df = pd.DataFrame(log_history)
    if df.empty:
        print("No GRPO log rows yet.")
        return df
    metric_cols = [col for col in df.columns if col in {"loss", "reward", "kl"} or "reward" in col or "rewards/" in col]
    metric_cols = [col for col in metric_cols if col != "epoch"]
    if not metric_cols or "step" not in df.columns:
        print("No plottable GRPO metrics. Columns:", list(df.columns))
        return df
    ax = df.dropna(subset=["step"]).plot(x="step", y=metric_cols, marker="o", figsize=(10, 5), title="GRPO training log")
    ax.grid(True, alpha=0.3)
    plt.show()
    return df


grpo_log_df = plot_grpo_log(grpo_log_history)
grpo_log_df.tail()

## Strict Evaluation: Base vs SFT vs GRPO

Evaluation does not use fallback. Invalid JSON counts as `parse_errors` and receives a penalty.

In [ ]:
from transformers import AutoModelForCausalLM


def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, device_map="auto")
    model.eval()
    model.config.use_cache = False
    return model


def load_adapter_model(path: str):
    model = AutoPeftModelForCausalLM.from_pretrained(path, device_map="auto")
    model.eval()
    model.config.use_cache = False
    return model


TEST_SEEDS = (1901, 1902)
active_model = None


def model_action_or_error(obs: dict[str, Any]) -> tuple[dict[str, Any] | None, str, str]:
    raw = generate_completion(active_model, prompt_from_observation(obs), do_sample=False)
    try:
        return parse_json_action(raw), raw, ""
    except Exception as exc:
        return None, raw, str(exc)


async def evaluate_model_policy(label: str, model, scenarios=SCENARIOS, seeds=TEST_SEEDS, max_steps: int = 20, verbose: bool = False):
    global active_model
    active_model = model
    rows = []
    for scenario_id in scenarios:
        for seed in seeds:
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                total_reward = 0.0
                steps = 0
                parse_errors = 0
                last_error = ""
                for step_idx in range(max_steps):
                    action, raw, error = model_action_or_error(obs)
                    if action is None:
                        parse_errors += 1
                        last_error = error
                        if verbose:
                            print(label, scenario_id, seed, f"step={step_idx:02d}", "PARSE_ERROR", raw[:240])
                        total_reward -= 1.0
                        break
                    if verbose:
                        print(label, scenario_id, seed, f"step={step_idx:02d}", action)
                    obs = await env.step(action)
                    steps = step_idx + 1
                    total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                    if obs.get("done"):
                        break
                rows.append({
                    "policy": label,
                    "scenario_id": scenario_id,
                    "seed": seed,
                    "total_reward": total_reward,
                    "done": bool(obs.get("done")),
                    "bookings": len(obs.get("booked_visits", [])),
                    "violations": len(obs.get("violations", [])),
                    "steps": steps,
                    "parse_errors": parse_errors,
                    "last_error": last_error,
                })
    return rows


async def evaluate_heuristic(label: str = "heuristic", scenarios=SCENARIOS, seeds=TEST_SEEDS, max_steps: int = 20):
    rows = []
    for scenario_id in scenarios:
        for seed in seeds:
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                total_reward = 0.0
                steps = 0
                for step_idx in range(max_steps):
                    action = heuristic_action(obs)
                    if action is None:
                        break
                    obs = await env.step(action)
                    steps = step_idx + 1
                    total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                    if obs.get("done"):
                        break
                rows.append({
                    "policy": label,
                    "scenario_id": scenario_id,
                    "seed": seed,
                    "total_reward": total_reward,
                    "done": bool(obs.get("done")),
                    "bookings": len(obs.get("booked_visits", [])),
                    "violations": len(obs.get("violations", [])),
                    "steps": steps,
                    "parse_errors": 0,
                    "last_error": "",
                })
    return rows


base_eval_model = load_base_model()
sft_eval_model = load_adapter_model(SFT_OUTPUT_DIR)
grpo_eval_model = load_adapter_model(GRPO_OUTPUT_DIR)

heuristic_eval = await evaluate_heuristic("heuristic")
base_eval = await evaluate_model_policy("base_model", base_eval_model, verbose=True)
sft_eval = await evaluate_model_policy("json_sft", sft_eval_model, verbose=True)
grpo_eval = await evaluate_model_policy("json_grpo", grpo_eval_model, verbose=True)

eval_df = pd.DataFrame(heuristic_eval + base_eval + sft_eval + grpo_eval)
eval_df

In [ ]:
def plot_policy_comparison(eval_df, title: str = "Base vs JSON SFT vs GRPO"):
    summary = (
        eval_df.groupby("policy", as_index=True)
        .agg(
            avg_reward=("total_reward", "mean"),
            completion_rate=("done", "mean"),
            avg_bookings=("bookings", "mean"),
            avg_violations=("violations", "mean"),
            avg_steps=("steps", "mean"),
            avg_parse_errors=("parse_errors", "mean"),
        )
        .sort_index()
    )
    axes = summary[["avg_reward", "completion_rate", "avg_bookings", "avg_violations", "avg_parse_errors"]].plot(
        kind="bar",
        subplots=True,
        layout=(3, 2),
        figsize=(11, 9),
        legend=False,
        title=title,
    )
    for ax in axes.ravel():
        ax.grid(axis="y", alpha=0.3)
        ax.set_xlabel("")
    plt.tight_layout()
    plt.show()
    return summary


comparison_summary = plot_policy_comparison(eval_df)
comparison_summary